# SPX close distribution → SPXW structures

Causal multi-session comparison of 10-point Call/Put debit verticals and 10/15/20-point butterflies. Entries use same-day SPXW exact BBO (long ask, short bid), with no midpoint substitution.

In [1]:
from datetime import datetime
from pathlib import Path
import runpy

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src/spx_spark').is_dir())
script = repo_root / 'docs/notebooks/spx-close-distribution-option-structures-2026-08-22.py'
module = runpy.run_path(str(script))
artifact = module['run_analysis'](write_outputs=False)
artifact['data_profile']

{'distribution_decisions': 140,
 'sessions': 14,
 'option_snapshot_contracts': 27962,
 'decision_snapshots_with_contracts': 130,
 'decision_snapshots': 140,
 'priced_candidates': 506,
 'rejections': {'butterfly_debit_fraction_exceeded': 468,
  'butterfly_debit_invalid': 1,
  'butterfly_leg_missing': 60,
  'butterfly_leg_quote_from_future': 39,
  'butterfly_leg_time_skew_exceeded': 14,
  'spread_leg_quote_from_future': 12,
  'vertical_leg_missing': 20}}

## Strict positive-objective policy

This is the closest comparison to an actionable gate: trade only when the existing risk-adjusted objective prefers the structure to cash.

In [2]:
artifact['first_trade_policy']

{'label': 'first_positive_objective_120_to_15',
 'horizon_minutes': None,
 'decisions': 14,
 'sessions': 14,
 'trades': 2,
 'trade_rate': 0.14285714285714285,
 'mean_pnl_per_decision_usd': 48.440000000002335,
 'session_bootstrap_95_mean_pnl_per_decision_usd': [-25.774285714285714,
  171.0942857142927],
 'mean_pnl_per_trade_usd': 339.08000000001635,
 'median_pnl_per_trade_usd': 339.08000000001635,
 'win_rate': 0.5,
 'total_pnl_usd': 678.1600000000327,
 'mean_predicted_minus_actual_usd': -130.50064707922664}

## Prior-session meta-policy diagnostic

After seven initial OOS sessions, family and horizon are selected only from prior realized sessions. This diagnostic deliberately bypasses the positive-objective gate to test whether the structure or the gate is the bottleneck.

In [3]:
artifact['prequential_structure_policy'], artifact['prequential_structure_audit']

({'label': 'prior_session_selected_family_horizon_diagnostic',
  'horizon_minutes': None,
  'decisions': 7,
  'sessions': 7,
  'trades': 6,
  'trade_rate': 0.8571428571428571,
  'mean_pnl_per_decision_usd': 335.80571428571636,
  'session_bootstrap_95_mean_pnl_per_decision_usd': [162.00985714285088,
   495.8685714285793],
  'mean_pnl_per_trade_usd': 391.7733333333358,
  'median_pnl_per_trade_usd': 476.4400000000164,
  'win_rate': 0.8333333333333334,
  'total_pnl_usd': 2350.640000000015,
  'mean_predicted_minus_actual_usd': -290.3460957881728},
 [{'session_date': '2026-08-12',
   'chosen_family': 'butterfly',
   'chosen_horizon_minutes': 60,
   'prior_sessions': 7,
   'prior_mean_pnl_per_decision_usd': 198.8685714285876,
   'candidate_id': '2026-08-12:60:butterfly:P:7740-7750-7760',
   'actual_pnl_usd': 525.4399999999855},
  {'session_date': '2026-08-13',
   'chosen_family': 'butterfly',
   'chosen_horizon_minutes': 60,
   'prior_sessions': 8,
   'prior_mean_pnl_per_decision_usd': 239.69

## Fixed-horizon research controls

In [4]:
fly_60 = next(row for row in artifact['best_priced_fixed_horizon_butterfly'] if row['horizon_minutes'] == 60)
vertical_30 = next(row for row in artifact['best_priced_fixed_horizon_vertical'] if row['horizon_minutes'] == 30)
{'butterfly_60m': fly_60, 'vertical_30m': vertical_30}

{'butterfly_60m': {'label': 'best_priced_butterfly_diagnostic',
  'horizon_minutes': 60,
  'decisions': 14,
  'sessions': 14,
  'trades': 13,
  'trade_rate': 0.9285714285714286,
  'mean_pnl_per_decision_usd': 267.33714285715195,
  'session_bootstrap_95_mean_pnl_per_decision_usd': [75.58128571428485,
   440.41492857144084],
  'mean_pnl_per_trade_usd': 287.9015384615483,
  'median_pnl_per_trade_usd': 435.44000000000364,
  'win_rate': 0.6923076923076923,
  'total_pnl_usd': 3742.7200000001276,
  'mean_predicted_minus_actual_usd': -164.44622778497128},
 'vertical_30m': {'label': 'best_priced_vertical_diagnostic',
  'horizon_minutes': 30,
  'decisions': 14,
  'sessions': 14,
  'trades': 13,
  'trade_rate': 0.9285714285714286,
  'mean_pnl_per_decision_usd': 137.45428571428127,
  'session_bootstrap_95_mean_pnl_per_decision_usd': [-28.50721428571721,
   321.9583571428444],
  'mean_pnl_per_trade_usd': 148.02769230768754,
  'median_pnl_per_trade_usd': -13.280000000010912,
  'win_rate': 0.38461538

## Conservative stress scenarios

In [5]:
artifact['prequential_stress']

[{'settlement_error_points': 0.0,
  'extra_entry_slippage_points': 0.0,
  'decisions': 7,
  'trades': 6,
  'mean_pnl_per_decision_usd': 335.8057142857164,
  'total_pnl_usd': 2350.6400000000144,
  'session_bootstrap_95_mean_pnl_per_decision_usd': [144.32185714285296,
   495.7435714285911]},
 {'settlement_error_points': 0.5,
  'extra_entry_slippage_points': 0.1,
  'decisions': 7,
  'trades': 6,
  'mean_pnl_per_decision_usd': 284.377142857145,
  'total_pnl_usd': 1990.6400000000147,
  'session_bootstrap_95_mean_pnl_per_decision_usd': [109.98285714286285,
   433.3185714285726]},
 {'settlement_error_points': 1.0,
  'extra_entry_slippage_points': 0.25,
  'decisions': 7,
  'trades': 6,
  'mean_pnl_per_decision_usd': 228.66285714285922,
  'total_pnl_usd': 1600.6400000000147,
  'session_bootstrap_95_mean_pnl_per_decision_usd': [67.06885714285109,
   366.62571428572426]},
 {'settlement_error_points': 2.0,
  'extra_entry_slippage_points': 0.5,
  'decisions': 7,
  'trades': 6,
  'mean_pnl_per_decis

## Independent temporal and payoff checks

In [6]:
candidates = artifact['candidates']
assert all(datetime.fromisoformat(source_at) <= datetime.fromisoformat(row['decision_at']) for row in candidates for source_at in row['quote_source_times'])
assert max(row['max_quote_age_seconds'] for row in candidates) <= 15.0
assert max(row['source_skew_seconds'] for row in candidates) <= 2.0
assert all(0.0 < row['debit_fraction'] <= 0.45 for row in candidates)
for row in candidates:
    settlement = row['settlement_proxy_spx']
    if row['family'] == 'vertical':
        long_strike, short_strike = row['strikes']
        intrinsic = (max(settlement - long_strike, 0.0) - max(settlement - short_strike, 0.0)) if row['right'] == 'C' else (max(long_strike - settlement, 0.0) - max(short_strike - settlement, 0.0))
    else:
        lower, center, upper = row['strikes']
        intrinsic = max(settlement - lower, 0.0) - 2.0 * max(settlement - center, 0.0) + max(settlement - upper, 0.0)
    expected = (intrinsic - row['net_debit'] - row['fees_points']) * 100.0
    assert abs(expected - row['actual_pnl_usd']) < 1e-7
{'candidate_count': len(candidates), 'max_quote_age_seconds': max(row['max_quote_age_seconds'] for row in candidates), 'max_source_skew_seconds': max(row['source_skew_seconds'] for row in candidates), 'payoff_checks': 'all_pass'}

{'candidate_count': 506,
 'max_quote_age_seconds': 4.354,
 'max_source_skew_seconds': 1.182,
 'payoff_checks': 'all_pass'}

## Takeaway

The strict gate produced only two trades in 14 sessions and remains unvalidated. The causal prior-session diagnostic repeatedly selected the 60-minute modal-center butterfly: six trades over seven evaluation sessions, five wins, +$2,350.64 total, and a session-bootstrap mean-PnL-per-decision interval above zero. This is a promising research candidate, not production evidence: the evaluation set is only seven days, the diagnostic bypasses the positive risk-objective gate, and expiry payoff uses the last RTH SPX observation as the settlement proxy. Directional verticals did not clear a stable lower-bound test.